# `GenVarLoader`

In [1]:
# Automatically reload code in notebook
%load_ext autoreload
%autoreload 2

import os
os.chdir(os.path.dirname(os.path.abspath('.')))
import pandas as pd
import polars as pl
import seqpro as sp
import genvarloader as gvl


import src.utils as utils
import src.config as config
import src.genvarloader as GVL
import src.onekg as og
# import src.pyensembl as PYE

/home/schilder/.conda/envs/gvl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import genvarloader as gvl
import numba as nb
import numpy as np
import polars as pl
import seqpro as sp
import pooch
from loguru import logger
from einops import rearrange
from tqdm.auto import tqdm

In [42]:

# GRCh38 chromosome 22 sequence
reference = pooch.retrieve(
    url="https://ftp.ensembl.org/pub/release-112/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.chromosome.22.fa.gz",
    known_hash="sha256:974f97ac8ef7ffae971b63b47608feda327403be40c27e391ee4a1a78b800df5",
    progressbar=True,
)
if not Path(f"{reference[:-3]}.bgz").exists():
    !gzip -dc {reference} | bgzip > {reference[:-3]}.bgz
reference = reference[:-3] + ".bgz"

# PLINK 2 files
variants = pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/1kGP.chr22.pgen",
    known_hash="md5:31aba970e35f816701b2b99118dfc2aa",
    progressbar=True,
    fname="1kGP.chr22.pgen",
)
pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/1kGP.chr22.psam",
    known_hash="md5:eefa7aad5acffe62bf41df0a4600129c",
    progressbar=True,
    fname="1kGP.chr22.psam",
)
pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/1kGP.chr22.pvar",
    known_hash="md5:5f922af91c1a2f6822e2f1bb4469d12b",
    progressbar=True,
    fname="1kGP.chr22.pvar",
)

# BED
bed_path = pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/chr22_egenes.bed",
    known_hash="md5:ccb55548e4ddd416d50dbe6638459421",
    progressbar=True,
)

/home/schilder/.conda/envs/gvl/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=179125) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


In [43]:
bed = gvl.with_length(gvl.read_bedlike(bed_path), 2**11)
bed.head()

chrom,chromStart,chromEnd,name,score,strand
str,i64,i64,str,f64,str
"""chr22""",41698475,41700523,"""ENSG00000167077""",null,"""+"""
"""chr22""",42834388,42836436,"""ENSG00000100266""",null,"""-"""
"""chr22""",20857959,20860007,"""ENSG00000099940""",null,"""+"""
"""chr22""",20706667,20708715,"""ENSG00000241973""",null,"""-"""
"""chr22""",49917143,49919191,"""ENSG00000184164""",null,"""+"""


In [44]:
tmp_dir = TemporaryDirectory(suffix=".gvl")
ds_path = tmp_dir.name
gvl.write(
    path=ds_path,
    bed=gvl.with_length(bed, 2**17),  # change region length to 131,072 bp
    variants=variants,
    overwrite=True,
)

/home/schilder/.conda/envs/gvl/lib/python3.12/tempfile.py:940: ResourceWarning: Implicitly cleaning up <TemporaryDirectory '/tmp/tmp98bn9b5m.gvl'>
  _warnings.warn(warn_message, ResourceWarning)
2025-05-17 23:57:56.562 | INFO     | genvarloader._dataset._write:write:76 - Writing dataset to /tmp/tmpizzf4jjt.gvl
2025-05-17 23:57:56.563 | INFO     | genvarloader._dataset._write:write:83 - Found existing GVL store, overwriting.
2025-05-17 23:57:56.573 | INFO     | genoray._pgen:_read_index:1077 - Loading genoray index.
2025-05-17 23:57:56.915 | INFO     | genvarloader._dataset._write:write:148 - Using 451 samples.
2025-05-17 23:57:56.916 | INFO     | genvarloader._dataset._write:write:154 - Writing genotypes.
Processing genotypes for 165 regions on contig chr22: 100%|██████████| 165/165 [00:03<00:00, 48.14 region/s]
2025-05-17 23:58:00.524 | INFO     | genvarloader._dataset._write:write:178 - Finished writing.


In [45]:
ds = (
    gvl.Dataset.open(ds_path, reference=reference)
    # .with_seqs("haplotypes")
    # .with_len(2**17)
)

2025-05-17 23:58:00.554 | INFO     | genvarloader._dataset._impl:open:215 - Loading reference genome into memory. This typically has a modest memory footprint (a few GB) and greatly improves performance.
2025-05-17 23:58:00.791 | INFO     | genvarloader._dataset._reconstruct:from_path:95 - Memory-mapping FASTA file for faster access.
Writing contig 22: 100%|██████████| 1/1 [00:00<00:00,  5.13it/s]
2025-05-17 23:58:01.001 | INFO     | genvarloader._dataset._reconstruct:from_path:298 - Loading variant data.
2025-05-17 23:58:01.022 | INFO     | genvarloader._dataset._impl:open:302 - Opened dataset:
GVL store at /tmp/tmpizzf4jjt.gvl
Is subset: False
# of regions: 165
# of samples: 451
Output length: ragged
Jitter: 0 (max: 0)
Deterministic: True
Sequence type: reference [haplotypes] annotated variants
Active tracks: None
Tracks available: None



## Annotation validation

In [46]:
annots = ds.regions.with_columns(chromEnd=pl.col("chromStart") + 1,
                                 score=pl.lit(1.0))
annots.head()

chrom,chromStart,chromEnd,name,score,strand
str,i64,i64,str,f64,str
"""chr22""",41633963,41633964,"""ENSG00000167077""",1.0,"""+"""
"""chr22""",42769876,42769877,"""ENSG00000100266""",1.0,"""-"""
"""chr22""",20793447,20793448,"""ENSG00000099940""",1.0,"""+"""
"""chr22""",20642155,20642156,"""ENSG00000241973""",1.0,"""-"""
"""chr22""",49852631,49852632,"""ENSG00000184164""",1.0,"""+"""


In [47]:
annot_ds = ds.write_annot_tracks({"3ss": annots}).with_tracks("3ss")
annot_ds

GVL store at /tmp/tmpizzf4jjt.gvl
Is subset: False
# of regions: 165
# of samples: 451
Output length: ragged
Jitter: 0 (max: 0)
Deterministic: True
Sequence type: reference [haplotypes] annotated variants
Active tracks: 3ss
Tracks available: 3ss

In [48]:
haps, tracks = annot_ds[1, :]
tracks.to_awkward()

<Array [[[[0, 0, 0, ..., 0, 0, 1], ...]], ...] type='451 * 1 * 2 * var * fl...'>

## Splicing validation

### Import UTR variant windows

500kb surrounding each ClinVar UTR variant.

In [ ]:
bed = gvl.read_bedlike("notebooks/snps_500kb_windows.bed")[:20]
bed.head()

### Import Haplotypes: Haplosaurus







### 1. Get transcript coordinates

Get transcript/exon coordinates from Ensembl (release 113) GTF:
https://ftp.ensembl.org/pub/release-113/gtf/homo_sapiens/

In [4]:
fn="https://ftp.ensembl.org/pub/release-113/gtf/homo_sapiens/Homo_sapiens.GRCh38.113.chr_patch_hapl_scaff.gtf.gz"

In [ ]:
import gffutils

dbfn = os.path.join(os.path.expanduser('~/.cache')  , 
                    os.path.basename(fn).replace('.gtf.gz', '.db'))
db = gffutils.create_db(fn, 
                        dbfn=dbfn, 
                        force=True, 
                        keep_order=True, 
                        merge_strategy='merge', 
                        sort_attribute_values=True, 
                        disable_infer_genes=True, 
                        disable_infer_transcripts=True)
annotations_db = gffutils.FeatureDB(db)


## `cyvcf2`

In [ ]:
import numpy as np
from cyvcf2 import VCF

def vcf_to_genotype_matrix(vcf_file, region):
    """
    Converts a VCF file to a genotype matrix for a specific region.

    Args:
        vcf_file (str): Path to the VCF file.
        region (str): Region to extract (e.g., "chr1:1000-2000").

    Returns:
        tuple: A tuple containing:
            - genotype_matrix (numpy.ndarray): Genotype matrix (samples x variants).
            - variant_ids (list): List of variant IDs.
            - sample_ids (list): List of sample IDs.
    """
    vcf = VCF(vcf_file)
    sample_ids = vcf.samples
    genotype_matrix = []
    variant_ids = []

    for variant in vcf(region):
        variant_ids.append(variant.ID)
        genotypes = [
            sum(gt) if gt != [-1, -1] else np.nan for gt in variant.genotypes
        ]
        genotype_matrix.append(genotypes)

    vcf.close()
    return np.array(genotype_matrix).T, variant_ids, sample_ids

vcf_file_path = "/grid/koo/home/schilder/projects/data/1KG/vcf/ALL.chr22.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz"
genotype_matrix, sample_names, variant_ids = vcf_to_genotype_matrix(vcf_file_path, "22:10664523-10684523")
print("Genotype Matrix:")
print(genotype_matrix)
print("\nSample Names:", sample_names)
print("\nVariant IDs:", variant_ids)

## `sgkit`

In [4]:
import sgkit as sg

In [5]:
ds = sg.simulate_genotype_call_dataset(n_variant=100, n_sample=50, missing_pct=.1)

In [ ]:
ds[['variant_allele', 'call_genotype']]


In [ ]:
ds.keys()

In [ ]:
df = ds.to_dataframe().reset_index()
df.head()

In [ ]:
# Create a new column by combining position and allele information
df['variant_id'] = df['variant_position'].astype(str) + df['variant_allele'].astype(str) + df['call_genotype'].astype(str)

# Create a ploidy-specific sample_id column
df['sample_id_ploidy'] = df['sample_id'].astype(str) + '_' + df['ploidy'].astype(str)

df.head()

Create a string representing the set of variants present in a given sample (one per ploid).

Then encode that string as an md5 hash for compression.

In [ ]:
import hashlib
df_agg = df.groupby('sample_id_ploidy')['variant_id'].apply(lambda x: ','.join(x)).reset_index()
df_agg['haplotype_hash'] = df_agg['variant_id'].apply(lambda x: hashlib.md5(x.encode()).hexdigest())
df_agg.head()


Create a hashmap for each haplotype hash to a list of sample_ids.

In [ ]:
# Group by haplotype_hash and get a list of sample_ids for each hash
haplotype_samples = df_agg.groupby('haplotype_hash')['sample_id_ploidy'].apply(list).reset_index()
print(haplotype_samples.shape)
haplotype_samples

In [ ]:
sg.display_genotypes(ds, max_variants=10, max_samples=10)

In [ ]:
sg.convert_call_to_index(ds).call_genotype_index.values


## `haptools`

`haptools` provides some convenient data structures and functions for working with haplotypes. BUT it does not actually create haplotype file for you.  
Users must generate these themselves beforehand and figure out how to convert it into the haptools-specific `.hap` file format.

PLINK can generate haplotype blocks, but this is a data-driven approach (using linkage disequilibrium-clumping information from a popuatlion) that does not produce one block of a given window size (as models like SpliceAI expect).
https://www.cog-genomics.org/plink/1.9/ld#blocks

In [12]:
from haptools import data

In [13]:
vcf_file_path="/grid/koo/home/schilder/projects/data/1KG/vcf/ALL.chr22.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz"
gt = data.GenotypesVCF(vcf_file_path) 

In [ ]:
print(gt.variants[:10])
print(gt.samples[:10])

In [ ]:
genotypes = gt.read(region="22:10664523-10764523")

In [ ]:
for i,line in enumerate(gt.__iter__(region="22:10664523-10764523")):
    print(line)
    if i > 5:
        break

In [ ]:
!haptools transform --region "22:10664523-10764523" {vcf_file_path} {vcf_file_path.replace('.vcf.gz', '.hap')}
